# 04 — Build a meaningful sampled PyG `HeteroData`

This notebook delegates to the repository's tested `build_pyg_export` implementation. It preserves typed node maps, relation-specific edge indices, reverse-edge identity, feature coverage, and deterministic learned fallbacks. The fixture is intentionally tiny; production/full exports remain sidecar-first and worker-only.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "public":
    REPO_ROOT = REPO_ROOT.parents[1]
elif REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = REPO_ROOT / "artifacts" / "cache" / "public-notebooks"
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})

try:
    import torch
    import torch_geometric
except ImportError as exc:
    raise RuntimeError("Install the notebook GNN environment with: uv sync --group notebooks --group gnn") from exc


In [ ]:
from manage_db.public_notebooks import build_sampled_pyg

if MODE != "fixture":
    raise RuntimeError("Public notebook live mode requires an explicitly staged bounded local subset; never point a laptop build at the full KG.")
PYG_ROOT = CACHE / "pyg-sample"
result = build_sampled_pyg(KG_ROOT, PYG_ROOT, max_nodes_per_type=100, max_edges_per_relation=200)
print(result)
manifest = json.loads((PYG_ROOT / "manifest.json").read_text())
print({"node_counts": result.node_counts, "edge_counts": manifest["edge_counts"]})


In [ ]:
from manage_db.public_notebooks import load_sampled_pyg

data = load_sampled_pyg(PYG_ROOT)
print(data)
print("node feature shapes:", {node_type: tuple(data[node_type].x.shape) for node_type in data.node_types})
print("edge types:", data.edge_types)
metadata = json.loads((PYG_ROOT / "heterodata" / "full_graph.metadata.json").read_text())
display(metadata.get("node_embedding_policy", {}))


The node maps define index↔biological-ID identity and must travel with tensors. Real vectors are used where joined; missing rows receive explicitly declared model-side learned fallbacks. A successful bounded build proves executability, not full-scale materialization, model quality, or biological validity.
